In [ ]:
# 安裝必要套件
!pip install yfinance pandas matplotlib numpy -q
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False


# 第 4 堂：均線與動能策略 — 完整回測


## MA Cross + RSI 策略回測

**進場規則：** 收盤價 > MA20 **且** RSI < 70
**出场：** 收盤價 < MA20


In [ ]:
TICKER = "2330.TW"
df = yf.Ticker(TICKER).history(period="2y")[['Close']].copy()
df['MA20'] = df['Close'].rolling(20).mean()
delta = df['Close'].diff()
gain = delta.where(delta > 0, 0).rolling(14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
df['RSI'] = 100 - (100 / (1 + gain / loss))
df['signal'] = 0
df.loc[(df['Close'] > df['MA20']) & (df['RSI'] < 70), 'signal'] = 1
df.loc[df['Close'] < df['MA20'], 'signal'] = 0
df['market_return'] = df['Close'].pct_change()
df['strategy_return'] = df['market_return'] * df['signal'].shift(1)
strategy_returns = df['strategy_return'].dropna()
market_returns = df['market_return'].dropna()
total_return = (1 + strategy_returns).prod() - 1
market_total = (1 + market_returns).prod() - 1
sharpe = strategy_returns.mean() / strategy_returns.std() * np.sqrt(252)
cum = (1 + strategy_returns).cumprod()
running_max = cum.cummax()
drawdown = (cum - running_max) / running_max
max_dd = drawdown.min()
print(f"策略總報酬：{total_return:.2%}")
print(f"買入持有報酬：{market_total:.2%}")
print(f"夏普比率：{sharpe:.2f}")
print(f"最大回撤：{max_dd:.2%}")
print(f"交易次數：{(df['signal'].diff() != 0).sum()}")


In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
ax1.plot(df['Close'], label='Close', alpha=0.8)
ax1.plot(df['MA20'], label='MA20', alpha=0.6)
buy_signals = df[df['signal'] == 1]
ax1.scatter(buy_signals.index, buy_signals['Close'], marker='^', color='green', s=80, label='Long Signal')
ax1.set_title(f'{TICKER} 價格 + MA20')
ax1.legend()
ax2.plot(df['RSI'], label='RSI(14)', color='purple')
ax2.axhline(70, color='red', linestyle='--', alpha=0.5)
ax2.axhline(30, color='green', linestyle='--', alpha=0.5)
ax2.set_title('RSI')
ax2.legend()
plt.tight_layout()
plt.show()
